# 🎥 Compiled Forensic CCTV Video Analyzer (Qwen3-VL-4B)
### Full 30 FPS + Hybrid GPU/RAM Overflow + Configurable Parameters

- **Compiled SDPA Attention**: Uses PyTorch C++ CUDA FlashAttention and Memory-Efficient Attention kernels.
- **Hybrid Memory Architecture**: Caps GPU VRAM to avoid OOM; seamlessly overflows token state and layer memory into System RAM.
- **Configurable Control Variables**: Video path, chunk duration, frame resolution (`target_size`), and memory pools are completely un-hardcoded.
- **Full 30 FPS Ingestion**: Captures every physical frame without temporal downsampling.
- **Structured Output**: Strictly emits `TIMESTAMP`, `TITLE` (one-line summary), `OVERALL DIRECTION OF MOTION`, `CONSOLIDATED FORENSIC ENTITY BREAKDOWN`, and `FINAL ACTION SUMMARY`.

In [1]:
# Cell 1: Setup & Load Qwen3-VL in Compiled Hybrid Mode (Run Once)
import os
import sys
import time
import tempfile
import urllib.parse
import requests
import cv2
import torch
import gc
from PIL import Image
from transformers import BitsAndBytesConfig, AutoModelForImageTextToText, AutoProcessor
from qwen_vl_utils import process_vision_info

# Ensure UTF-8 output on Windows
if sys.platform == "win32":
    try:
        sys.stdout.reconfigure(encoding="utf-8")
    except Exception:
        pass

# =========================================================================
# ⚙️ HYBRID HARDWARE CONFIGURATION
# =========================================================================
MODEL_PATH = os.path.abspath(r"models\Qwen3-VL-4B-Instruct")
GPU_VRAM_LIMIT = "3200MiB"  # Caps VRAM so CUDA kernels and activations have headroom
SYSTEM_RAM_POOL = "24GiB"   # Large system RAM pool for overflow
OFFLOAD_DIR = "offload"

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    try:
        torch.backends.cuda.enable_flash_sdp(True)
        torch.backends.cuda.enable_mem_efficient_sdp(True)
        torch.backends.cuda.enable_math_sdp(True)
    except Exception:
        pass

print(f"[*] Loading model from: {MODEL_PATH}")
print(f"[*] Hybrid limits: GPU={GPU_VRAM_LIMIT} | System RAM={SYSTEM_RAM_POOL}")

os.makedirs(OFFLOAD_DIR, exist_ok=True)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_PATH)
max_memory = {0: GPU_VRAM_LIMIT, "cpu": SYSTEM_RAM_POOL}

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map="auto",
    max_memory=max_memory,
    offload_folder=OFFLOAD_DIR,
    attn_implementation="sdpa"  # Compiled PyTorch C++ CUDA FlashAttention / Mem-Efficient kernels
)

print(f"[✓] Qwen3-VL loaded in {time.time() - t0:.1f}s!")
print(f"[*] Primary device: {model.device if hasattr(model, 'device') else 'cuda:0'}")
print(f"[*] Active VRAM: {torch.cuda.memory_allocated() / (1024*1024):.1f} MB")


c:\Users\halda\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[*] Loading model from: c:\Users\halda\OneDrive\Desktop\CCTV\models\Qwen3-VL-4B-Instruct


W1104 17:19:48.148000 26220 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading weights: 100%|██████████| 713/713 [00:11<00:00, 60.76it/s] 


[✓] Qwen3-VL-4B loaded on GPU in 18.6s!
[*] Active VRAM allocated: 2775.3 MB / 4096 MB


In [2]:
# Cell 2: Forensic Prompts & Dynamic Chunking Engine

# Prompt for analyzing an individual 30-FPS chunk
FORENSIC_CHUNK_PROMPT = """
You are an expert CCTV Forensic Video Analyst. Analyze this 30-FPS video chunk with maximum forensic precision.

CRITICAL INSTRUCTIONS:
1. Timestamp: Note on-screen camera timestamp if visible.
2. POSITIVE IDENTIFICATION ONLY: Describe ONLY moving subjects (people, vehicles, animals, objects) and dynamic actions that are ACTUALLY PRESENT in the video. NEVER list absent entities (do NOT say \"no animals\", \"no weapons\", \"no pedestrians\", \"no visible items\", etc.).
3. Ignore Background: Do NOT describe stationary background elements (walls, road, stationary gates, trees, pavement).
4. For every moving entity present, describe in deep detail:
   - Entity Category & Type: Person (gender, build, clothing colors/types, footwear, carried items), Vehicle (type, body color, markings, driver/rider), Animal (species, fur pattern, color).
   - Chunk Motion Direction: Exact direction and movement trajectory during this chunk (e.g. entering from right side, moving toward gate, turning left, heading away from camera).
""".strip()

# Master Synthesis Prompt: merges all chunk directions into one overall trajectory; outputs timestamp ONLY ONCE; stops only on EOS
SYNTHESIS_PROMPT_TEMPLATE = """
You are a Lead CCTV Forensic Security Analyst. Below is a chronological log of observations extracted across consecutive 10-second 30-FPS chunks of a single CCTV event:

{timeline}

TASK: Synthesize the above chunk observations into a single, clean, cohesive forensic report.

CRITICAL FORMATTING RULES:
Output MUST strictly follow this exact structure and field names:

TIMESTAMP: <On-screen camera timestamp if visible, stated EXACTLY ONCE>
TITLE: <A short, concise one-line summary of the entire clip>
OVERALL DIRECTION OF MOTION: <Single, unified trajectory combining directional movements across all chunks from start to finish>
CONSOLIDATED FORENSIC ENTITY BREAKDOWN:
<For every moving entity observed (person, vehicle, animal): deep visual description (colors, build, clothing, vehicle finish/type, carried items, distinguishing features)>
FINAL ACTION SUMMARY:
<Coherent narrative of the entire dynamic event from start to finish>

STRICT PROHIBITIONS:
- Do NOT output semantic search tags or keyword lists.
- Do NOT output chunk headers (no 'Window [0.0s - 10.0s]', etc.).
- Do NOT list absent entities (never say 'no animals', 'no pedestrians', etc.).
- Do NOT write bureaucratic report headers (no 'Lead Analyst:', 'Report ID:', etc.).
- Output ONLY the final structured text.
""".strip()

# Single-pass prompt used when video is <= chunk_duration_sec
SINGLE_PASS_PROMPT = """
You are an expert CCTV Forensic Video Analyst. Analyze this 30-FPS video clip with maximum forensic precision.

CRITICAL FORMATTING RULES:
Output MUST strictly follow this exact structure and field names:

TIMESTAMP: <On-screen camera timestamp if visible, stated EXACTLY ONCE>
TITLE: <A short, concise one-line summary of the entire clip>
OVERALL DIRECTION OF MOTION: <Single, unified trajectory detailing overall movement from point of origin to destination/exit>
CONSOLIDATED FORENSIC ENTITY BREAKDOWN:
<For every moving entity present (person, vehicle, animal): deep visual description including gender, build, clothing, vehicle finish/type, carried items, distinguishing features>
FINAL ACTION SUMMARY:
<Coherent narrative of the entire dynamic event from start to finish>

STRICT PROHIBITIONS:
- POSITIVE IDENTIFICATION ONLY: Describe ONLY moving subjects and dynamic actions that are ACTUALLY PRESENT. Never describe absent entities (never say \"no animals\", \"no weapons\", etc.).
- Ignore Background: Do NOT describe stationary background elements (walls, road, stationary gates, trees, pavement).
- Do NOT output semantic search tags or keyword lists.
- Output ONLY the final structured text.
""".strip()


def download_if_url(video_source):
    parsed = urllib.parse.urlparse(video_source)
    if parsed.scheme in ["http", "https"]:
        print(f"[*] Downloading video from: {video_source}...")
        resp = requests.get(video_source, stream=True, timeout=30)
        resp.raise_for_status()
        suffix = os.path.splitext(parsed.path)[1] or ".mp4"
        temp_fd, temp_file = tempfile.mkstemp(suffix=suffix)
        with os.fdopen(temp_fd, "wb") as f:
            for chunk in resp.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
        return temp_file, True
    return video_source, False


def parse_resolution(res_input):
    if isinstance(res_input, (tuple, list)):
        return (int(res_input[0]), int(res_input[1]))
    res_str = str(res_input).lower().replace("p", "").strip()
    if "x" in res_str:
        w, h = res_str.split("x")
        return (int(w), int(h))
    elif "," in res_str:
        w, h = res_str.split(",")
        return (int(w), int(h))
    elif res_str == "720": return (1280, 720)
    elif res_str == "480": return (854, 480)
    elif res_str == "360": return (640, 360)
    elif res_str == "270": return (480, 270)
    return (640, 360)


def read_all_video_frames(video_path, target_size=(640, 360)):
    """Reads all actual frames sequentially to avoid inaccurate MP4 metadata headers."""
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    if not fps or fps <= 0 or fps > 120:
        fps = 30.0
    target_w, target_h = parse_resolution(target_size)
    frames = []
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        resized = cv2.resize(frame, (target_w, target_h), interpolation=cv2.INTER_AREA)
        rgb_frame = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
        frames.append(Image.fromarray(rgb_frame))
    cap.release()
    return frames, fps


def analyze_single_chunk(frames, prompt_to_use, chunk_label="", max_new_tokens=4096):
    if not frames or len(frames) == 0:
        return ""
    prompt_text = f"[{chunk_label}]\n" + prompt_to_use if chunk_label else prompt_to_use
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "video", "video": frames, "fps": 30.0},
                {"type": "text", "text": prompt_text}
            ]
        }
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    primary_device = model.device if hasattr(model, 'device') else 'cuda:0'
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to(primary_device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs, 
            max_new_tokens=max_new_tokens,
            eos_token_id=processor.tokenizer.eos_token_id
        )
        trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
        response = processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

    del inputs, image_inputs, video_inputs, generated_ids
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return response.strip()


def synthesize_chunks(chunk_results, max_new_tokens=4096):
    valid_results = [res for res in chunk_results if res['description'].strip()]
    if not valid_results:
        return "No moving subjects detected across any chunk."

    timeline_str = "\n\n".join([
        f"--- Event Window {res['window']} ---\n{res['description']}" 
        for res in valid_results
    ])
    prompt = SYNTHESIS_PROMPT_TEMPLATE.format(timeline=timeline_str)
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    primary_device = model.device if hasattr(model, 'device') else 'cuda:0'
    inputs = processor(text=[text], padding=True, return_tensors="pt").to(primary_device)

    with torch.no_grad():
        out = model.generate(
            **inputs, 
            max_new_tokens=max_new_tokens,
            eos_token_id=processor.tokenizer.eos_token_id
        )
        res = processor.batch_decode(out[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]

    del inputs, out
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return res.strip()


def analyze_video_hierarchical(
    video_source,
    chunk_duration_sec=10.0,
    target_size=(640, 360),
    max_new_tokens=4096
):
    """
    Main Analyzer:
    - Maximum chunk_duration_sec at full 30 FPS.
    - Reads frames sequentially to avoid inaccurate MP4 header issues.
    - Tracks per-chunk direction, then synthesizes into overall direction.
    - Returns ONLY the clean final text with one timestamp.
    - Compiled SDPA attention + Hybrid memory overflow.
    """
    local_path, is_temp = download_if_url(video_source)
    try:
        all_frames, fps = read_all_video_frames(local_path, target_size=target_size)
        total_frames = len(all_frames)
        if total_frames == 0:
            return "Error: Could not extract any valid video frames from video."

        duration_sec = total_frames / fps
        frames_per_chunk = max(int(fps * chunk_duration_sec), 1)

        chunks = [
            all_frames[i : i + frames_per_chunk]
            for i in range(0, total_frames, frames_per_chunk)
        ]
        chunks = [c for c in chunks if len(c) >= 2]

        if len(chunks) <= 1:
            return analyze_single_chunk(
                frames=all_frames,
                prompt_to_use=SINGLE_PASS_PROMPT,
                chunk_label=f"0.0s - {duration_sec:.1f}s",
                max_new_tokens=max_new_tokens
            )

        chunk_results = []
        for i, chunk_frames in enumerate(chunks):
            start_f = i * frames_per_chunk
            end_f = start_f + len(chunk_frames)
            window_label = f"{start_f / fps:.1f}s - {end_f / fps:.1f}s"

            desc = analyze_single_chunk(
                frames=chunk_frames,
                prompt_to_use=FORENSIC_CHUNK_PROMPT,
                chunk_label=window_label,
                max_new_tokens=max_new_tokens
            )
            if desc.strip():
                chunk_results.append({"chunk_idx": i + 1, "window": window_label, "description": desc})

        final_output_text = synthesize_chunks(chunk_results, max_new_tokens=max_new_tokens)
        return final_output_text

    finally:
        if is_temp and os.path.exists(local_path):
            try:
                os.remove(local_path)
            except Exception:
                pass


### 🎬 Run Forensic Video Analysis
Provide any local video path or online video URL below:

In [ ]:
# Cell 3: Execute Analysis with Configurable Variables (Freely change any value)

# =========================================================================
# ⚙️ CONFIGURABLE CONTROL VARIABLES
# =========================================================================
VIDEO_SOURCE = r"test\clips\event_clip_05_04m50s_to_04m59s.mp4"
# Or use any web URL: VIDEO_SOURCE = "https://example.com/cctv_event.mp4"

# Maximum chunk duration in seconds at full 30 FPS
CHUNK_DURATION_SEC = 10.0

# Target Frame Resolution: '(640, 360)', '(854, 480)', '(1280, 720)', or '(480, 270)'
TARGET_SIZE = (640, 360)

# Maximum tokens generated (stops automatically on <|im_end|>)
MAX_NEW_TOKENS = 4096

# =========================================================================
# 🎬 RUN FORENSIC ANALYSIS
# =========================================================================
t0 = time.time()
final_output = analyze_video_hierarchical(
    video_source=VIDEO_SOURCE,
    chunk_duration_sec=CHUNK_DURATION_SEC,
    target_size=TARGET_SIZE,
    max_new_tokens=MAX_NEW_TOKENS
)

print("=" * 70)
print("📋 FINAL FORENSIC CCTV OUTPUT:")
print("=" * 70)
print(final_output)
print("=" * 70)
print(f"[*] Analysis completed in {time.time() - t0:.1f}s")
